# Costruzione decrizione automatica svg funzioni

In [232]:
import xml.etree.ElementTree as ET
import re
import os
import xml.dom.minidom
import webbrowser

## Parsing ed inserimento delle informazioni all'interno delle strutture dati

In [233]:
def parsing_svg(nome_file_svg):
    tree = ET.parse(nome_file_svg)
    root_svg = tree.getroot()

    #find element with id="funzioni1"
    for child in root_svg:
        if child.attrib['id'] == 'funzioni1':
            funzioni1 = child

    componenti = {}
    for child in funzioni1:
        if not re.search('title', child.attrib['id']):
            nome_componente = child.attrib['id']
            componenti[nome_componente] = []

            componente = {}
            for child2 in child:
                if not re.search('title', child2.attrib['id']):
                    componente[child2.attrib['id']] = []
                
                    #get tspan
                    for child3 in child2:
                        if child3.tag == '{http://www.w3.org/2000/svg}tspan':
                            componente[child2.attrib['id']] = child3.text

            componenti[nome_componente].append(componente)

    return componenti

def information_retrival_by_svg_tree(componenti):
    nome_insieme_dominio = ''
    elementi_dominio = []

    nome_insieme_codominio = ''
    elementi_codominio = []

    link = {}
    for componente in componenti:
        if (componente == 'dominio'):
            for child in componenti[componente]:
                for child2 in child:
                    if re.search('nome', child2):
                        nome_insieme_dominio = child[child2]
                    elif not re.search('insieme', child2):
                        elementi_dominio.append(child2)

        elif (componente == 'codominio'):
            
            for child in componenti[componente]:
                for child2 in child:
                    if re.search('nome', child2):
                        nome_insieme_codominio = child[child2]
                    elif not re.search('insieme', child2):
                        elementi_codominio.append(child2)

        elif (re.search('link', componente)):
            componenti_split = componente.split('-')
            if componenti_split[1] in link:
                link[componenti_split[1]] = link[componenti_split[1]] + ' ' + componenti_split[2]
            else:
                link[componenti_split[1]] = componenti_split[2]

    return nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link

## Generazione pattern-template

### Dialogue acts

I dialogue acts utilizzati sono i seguenti
- <b>DS:opening</b>: saluti iniziali (es. ciao, buongiorno, ...)
- <b>Ta:setQuestion</b>: domanda specifica (es. quante elementi nel dominio, ...)
- <b>Ta:request</b>: richiesta generica (es. parlami di, descrivimi, ...)
- <b>Ta:propositionalQuestion</b>: domanda che richiede una risposta che confermi o confuti una dichiarazione (es. è vero che ...)

Variabili:
- Dialogue act
- elemento1
- elemento2
- insieme
- topic

In [234]:
vars = {
    'dialogue_act': 'none',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': 'none',
    'topic': 'none',
}

dialogue_acts = [
    'setQuestionAnswer',
    'requestAnswer',
    'propositionalQuestionAnswer',
]

### Ta:setQuestion

Esempio: 
- Quanti elementi nel dominio?
- A cosa è associato x?
- ...

#### 1. Caso in cui l'elemento del dominio non è collegato a nulla nel codominio

In [235]:
def generateSetQuestion1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    for elemento in elementi_dominio:
        trovato = False
        for key in link:
            if key == elemento:
                trovato = True
                break

        if not trovato:
            domande = [
                '* associato * ' + elemento + ' *',
                '* associata * ' + elemento + ' *',
                '* collegato * ' + elemento + ' *',
                '* collegata * ' + elemento + ' *',
                '* legato * ' + elemento + ' *',
                '* legata * ' + elemento + ' *',
                '* relazione * ' + elemento + ' *',
                '* immagine * ' + elemento + ' *',
                '* funzione * ' + elemento + ' *',
                '* f * ' + elemento + ' *',
                '* quali frecce * ' + elemento + ' *',

                '* ' + elemento + ' * associato *',
                '* ' + elemento + ' * associata *',
                '* ' + elemento + ' * collegato *',
                '* ' + elemento + ' * collegata *',
                '* ' + elemento + ' * legato *',
                '* ' + elemento + ' * legata *',
                '* ' + elemento + ' * relazione *',
                '* ' + elemento + ' * immagine *',
                '* ' + elemento + ' * funzione *',
                '* ' + elemento + ' * f *',
                '* ' + elemento + ' * quali frecce *',
            ]

            # variabili
            vars = {
                'dialogue_act': 'setQuestion',
                'elemento1': elemento,
                'elemento2': 'none',
                'insieme': nome_insieme_dominio,
                'topic': 'funzione',
            }

            # contenuto testuale
            text = ''
            for var in vars:
                text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

            text += elemento + ' non è associato a nessun elemento del codominio'

            # immagine
            text += '<image>insiemi/' + filename + '</image>'
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

            for domanda in domande:
                domanda_uppercase = domanda.upper()            

                setQuestionAnswer[domanda_uppercase] = text

    return setQuestionAnswer

#### 2. Caso in cui l'elemento del codominio non è collegato a nulla nel dominio

In [236]:
def generateSetQuestion2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    for elemento in elementi_codominio:
        trovato = False
        for key in link:
            if link[key] == elemento:
                trovato = True
                break

        if not trovato:
            domande = [
                '* ' + elemento + ' * associato *',
                '* ' + elemento + ' * associata *',
                '* ' + elemento + ' * collegato *',
                '* ' + elemento + ' * collegata *',
                '* ' + elemento + ' * legato *',
                '* ' + elemento + ' * legata *',
                '* ' + elemento + ' * relazione *',
                '* ' + elemento + ' * controimmagine *',
                '* ' + elemento + ' * funzione inversa *',
                '* ' + elemento + ' * f^-1 *',
                '* ' + elemento + ' * f-1 *',
                '* ' + elemento + ' * quali frecce *',
                
                '* associato * ' + elemento + ' *',
                '* associata * ' + elemento + ' *',
                '* collegato * ' + elemento + ' *',
                '* collegata * ' + elemento + ' *',
                '* legato * ' + elemento + ' *',
                '* legata * ' + elemento + ' *',
                '* relazione * ' + elemento + ' *',
                '* controimmagine * ' + elemento + ' *',
                '* funzione inversa * ' + elemento + ' *',
                '* f^-1 * ' + elemento + ' *',
                '* f-1 *' + elemento + ' *',
                '* quali frecce * ' + elemento + ' *',
            ]

            # variabili
            vars = {
                'dialogue_act': 'setQuestion',
                'elemento1': elemento,
                'elemento2': 'none',
                'insieme': nome_insieme_codominio,
                'topic': 'funzione_inversa',
            }

            text = ''
            for var in vars:
                text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

            # contenuto testuale
            text += elemento + ' non è associato a nessun elemento del dominio'

            # immagine
            text += '<image>insiemi/' + filename + '</image>'
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

            for domanda in domande:
                domanda_uppercase = domanda.upper()
                
                setQuestionAnswer[domanda_uppercase] =  text

    return setQuestionAnswer

#### 3. Numero di elementi nel dominio

In [237]:
def generateSetQuestion3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    domande = [
        '* ' + nome_insieme_dominio + ' *',
        '* dominio *',
        '* primo cerchio *',
        '* primo insieme *',
        '* cerchio uno *',
        '* insieme uno *',

    ]

    # variabili
    vars = {
        'dialogue_act': 'setQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    if len(elementi_dominio) == 0:
        text += 'Il primo cerchio (nel dominio) è vuoto.'
    elif len(elementi_dominio) == 1:
        text += 'Nel primo cerchio (nel dominio) c\'è un solo elemento: ' + elementi_dominio[0] + '.'
    else:
        text += 'Nel primo cerchio (nel dominio) ci sono ' + str(len(elementi_dominio)) + ' elementi: '
        for elemento in elementi_dominio:
            text += elemento + ', '
        text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        setQuestionAnswer[domanda_uppercase] = text

    return setQuestionAnswer

#### 4. Numero di elementi nel codominio

In [238]:
def generateSetQuestion4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    domande = [
        '* ' + nome_insieme_codominio + ' *',
        '* codominio *',
        '* secondo cerchio *',
        '* secondo insieme *',
        '* cerchio due *',
        '* insieme due *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'setQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    if len(elementi_codominio) == 0:
        text += 'Il secondo cerchio (nel codominio) è vuoto.'
    elif len(elementi_codominio) == 1:
        text += 'Nel secondo cerchio (nel codominio) c\'è un solo elemento: ' + elementi_codominio[0] + '.'
    else:
        text += 'L\'insieme ' + nome_insieme_codominio + ' è composto da ' + str(len(elementi_codominio)) + ' elementi: '
        for elemento in elementi_codominio:
            text += elemento + ', '
        text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        setQuestionAnswer[domanda_uppercase] = text

    return setQuestionAnswer

#### 5. A cosa è collegato l'elemento del dominio

In [239]:
def generateSetQuestion5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    for elemento in link:
        domande = [
            '* elemento * collegato * ' + elemento + ' *',
            '* collegato * ' + elemento + ' *',
            '* ' + elemento + ' * collegato *',
            '* collegata * ' + elemento + ' *',
            '* ' + elemento + ' * collegata *',
            '* ' + elemento + ' *',
            '* ' + nome_insieme_codominio + ' * ' + elemento + ' *',
            '* ' + elemento + ' * ' + nome_insieme_codominio + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'setQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_dominio,
            'topic': 'funzione',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        if len(link[elemento].split(' ')) > 1:
            text += 'L\'elemento ' + elemento + ' dell\'insieme ' + nome_insieme_dominio + ' è collegato agli elementi '

            num_elementi = len(link[elemento].split(' '))
            for elemento_codominio in link[elemento].split(' '):
                text += elemento_codominio
                num_elementi -= 1

                if num_elementi > 1:
                    text += ', '
                elif num_elementi == 1:
                    text += ' e '

            text = text + ' dell\'insieme ' + nome_insieme_codominio + '.'
        else :
            text += 'L\'elemento ' + elemento + ' dell\'insieme ' + nome_insieme_dominio + ' è collegato all\'elemento ' + link[elemento] + ' dell\'insieme ' + nome_insieme_codominio + '.'
        
        # immagine
        text += '<image>insiemi/' + filename + '</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + link[elemento] + '</svgElement>'
        
        for domanda in domande:
            domanda_uppercase = domanda.upper()
            setQuestionAnswer[domanda_uppercase] = text

    return setQuestionAnswer

#### 6. A cosa è collegato l'elemento del codominio

In [240]:
def generateSetQuestion6(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    for elemento in link:
        els = []
        if len(link[elemento].split(' ')) > 1:
            for el in link[elemento].split(' '):
                els.append(el)
        else:
            els.append(link[elemento])
        
        for el in els:
            domande = [
            '* collegato * ' + el + ' *',
            '* ' + el + ' * collegato *',
            '* collegata * ' + el + ' *',
            '* ' + el + ' * collegata *',
            '* ' + el + ' *'
            ]

            # variabili
            vars = {
                'dialogue_act': 'setQuestion',
                'elemento1': el,
                'elemento2': 'none',
                'insieme': nome_insieme_codominio,
                'topic': 'funzione_inversa',
            }

            text = ''
            for var in vars:
                text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

            # conteggio dei collegamenti dal dominio all'elemento del codominio
            count = 0
            for key in link:
                if link[elemento] == link[key]:
                    count += 1

            # contenuto testuale
            if count > 1:
                num_elementi = count
                for elemento_dominio in elementi_dominio:
                    if link[elemento] == link[elemento_dominio]:
                        text += elemento_dominio
                        num_elementi -= 1

                        if num_elementi > 1:
                            text += ', '
                        elif num_elementi == 1:
                            text += ' e '

                text += ' sono collegati a ' + el + '.'
                # rendi la prima lettera maiuscola
                text = text[0].upper() + text[1:]

                text += '<image>insiemi/' + filename + '</image>'

                for elemento_dominio in elementi_dominio:
                    if link[elemento] == link[elemento_dominio]:
                        text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento_dominio + '-' + link[elemento] + '</svgElement>'
            else :
                text += elemento + ' è collegato a ' + el + '.'

                # immagine
                text += '<image>insiemi/' + filename + '</image>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + link[elemento] + '</svgElement>'

            for domanda in domande:
                domanda_uppercase = domanda.upper()
                setQuestionAnswer[domanda_uppercase] = text

    return setQuestionAnswer

#### Creazione delle setQuestion

In [241]:
def generateSetQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setQuestionAnswer = {}

    setQuestionAnswer.update(generateSetQuestion1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setQuestionAnswer.update(generateSetQuestion2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setQuestionAnswer.update(generateSetQuestion3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setQuestionAnswer.update(generateSetQuestion4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setQuestionAnswer.update(generateSetQuestion5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setQuestionAnswer.update(generateSetQuestion6(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))

    return setQuestionAnswer

### Ta:request

Esempio:
- parlami del dominio
- descrivimi il codominio

#### 1. Parlami del dominio

In [242]:
def generateRequest1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    requestAnswer = {}

    domande = [
        '* parlami * dominio *',
        '* parlami * ' + nome_insieme_dominio + ' *',
        '* parlami * primo cerchio *',
        '* parlami * primo insieme *',

        '* parla * dominio *',
        '* parla * ' + nome_insieme_dominio + ' *',
        '* parla * primo cerchio *',
        '* parla * primo insieme *',
        '* spiegami * dominio *',
        '* spiegami * ' + nome_insieme_dominio + ' *',
        '* spiagami * primo cerchio *',
        '* spiegami * primo insieme *',

        '* spiega * dominio *',
        '* spiega * ' + nome_insieme_dominio + ' *',
        '* spiega * primo cerchio *',
        '* spiega * primo insieme *',

        '* descrivimi * dominio *',
        '* descrivimi * ' + nome_insieme_dominio + ' *',
        '* descrivimi * primo cerchio *',
        '* descrivimi * primo insieme *',

        '* descrivi * dominio *',
        '* descrivi * ' + nome_insieme_dominio + ' *',
        '* descrivi * primo cerchio *',
        '* descrivi * primo insieme *',

        '* raccontami * dominio *',
        '* raccontami * ' + nome_insieme_dominio + ' *',
        '* raccontami * primo cerchio *',
        '* raccontami * primo insieme *',

        '* racconta * dominio *',
        '* racconta * ' + nome_insieme_dominio + ' *',
        '* racconta * primo cerchio *',
        '* racconta * primo insieme *',

        '* elementi * dominio *',
        '* elementi * ' + nome_insieme_dominio + ' *',
        '* elementi * insieme * ' + nome_insieme_dominio + ' *',
        '* elementi * cerchio * ' + nome_insieme_dominio + ' *',
        '* elementi * primo cerchio *',
        '* elementi * primo insieme *',
    ]


    # variabili
    vars = {
        'dialogue_act': 'request',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if len(elementi_dominio) == 0:
        text += 'Il primo cerchio (nel dominio) è vuoto.'
    elif len(elementi_dominio) == 1:
        text += 'Nel primo cerchio (nel dominio) c\'è un solo elemento: ' + elementi_dominio[0] + '.'
    else:
        text += 'L\'insieme ' + nome_insieme_dominio + ' è composto da ' + str(len(elementi_dominio)) + ' elementi: '
    for elemento in elementi_dominio:
        text += elemento + ', '
    text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        requestAnswer[domanda_uppercase] = text

    return requestAnswer

#### 2. Parlami del codominio

In [243]:
def generateRequest2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    requestAnswer = {}

    domande = [
        '* parlami * codominio *',
        '* parlami * ' + nome_insieme_codominio + ' *',
        '* parlami * secondo cerchio *',
        '* parlami * secondo insieme *',

        '* parla * codominio *',
        '* parla * ' + nome_insieme_codominio + ' *',
        '* parla * secondo cerchio *',
        '* parla * secondo insieme *',

        '* spiegami * codominio *',
        '* spiegami * ' + nome_insieme_codominio + ' *',
        '* spiagami * secondo cerchio *',
        '* spiegami * secondo insieme *',

        '* spiega * codominio *',
        '* spiega * ' + nome_insieme_codominio + ' *',
        '* spiega * secondo cerchio *',
        '* spiega * secondo insieme *',

        '* descrivimi * codominio *',
        '* descrivimi * ' + nome_insieme_codominio + ' *',
        '* descrivimi * secondo cerchio *',
        '* descrivimi * secondo insieme *',

        '* descrivi * codominio *',
        '* descrivi * ' + nome_insieme_codominio + ' *',
        '* descrivi * secondo cerchio *',
        '* descrivi * secondo insieme *',

        '* raccontami * codominio *',
        '* raccontami * ' + nome_insieme_codominio + ' *',
        '* raccontami * secondo cerchio *',
        '* raccontami * secondo insieme *',

        '* racconta * codominio *',
        '* racconta * ' + nome_insieme_codominio + ' *',
        '* racconta * secondo cerchio *',
        '* racconta * secondo insieme *',

        '* elementi * codominio *',
        '* elementi * ' + nome_insieme_codominio + ' *',
        '* elementi * insieme * ' + nome_insieme_codominio + ' *',
        '* elementi * cerchio * ' + nome_insieme_codominio + ' *',
        '* elementi * secondo cerchio *',
        '* elementi * secondo insieme *',
    ]

    vars = {
        'dialogue_act': 'request',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_codominio + '</set></think>'

    # contenuto testuale
    if len(elementi_codominio) == 0:
        text += 'Il secondo cerchio (nel codominio) è vuoto.'
    elif len(elementi_codominio) == 1:
        text += 'Nel secondo cerchio (nel codominio) c\'è un solo elemento: ' + elementi_codominio[0] + '.'
    else:
        text += 'Nel secondo cerchio (codominio) ci sono ' + str(len(elementi_codominio)) + ' elementi: '
    for elemento in elementi_codominio:
        text += elemento + ', '
    text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        requestAnswer[domanda_uppercase] = text

    return requestAnswer

#### 3. Parlami degli insiemi

In [244]:
def generateRequest3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    requestAnswer = {}

    domande = [
        '* insiemi *',
        '* cerchi *',

        '* parlami * dominio * codominio *',
        '* parlami * codominio * dominio *',
        '* parlami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* parlami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* parlami * insiemi *',
        '* parlami * cerchi *',
        '* parlami * primo cerchio * secondo cerchio *',
        '* parlami * primo insieme * secondo insieme *',
        '* parlami * cerchio uno * cerchio due *',
        '* parlami * insieme uno * insieme due *',
        '* parlami * cerchio 1 * cerchio 2 *',
        '* parlami * insieme 1 * insieme 2 *',
        '* parlami * cerchio due * cerchio uno *',
        '* parlami * insieme due * insieme uno *',
        '* parlami * cerchio 2 * cerchio 1 *',
        '* parlami * insieme 2 * insieme 1 *',

        '* parla * dominio * codominio *',
        '* parla * codominio * dominio *',
        '* parla * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* parla * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* parla * insiemi *',
        '* parla * cerchi *',
        '* parla * primo cerchio * secondo cerchio *',
        '* parla * primo insieme * secondo insieme *',
        '* parla * cerchio uno * cerchio due *',
        '* parla * insieme uno * insieme due *',
        '* parla * cerchio 1 * cerchio 2 *',
        '* parla * insieme 1 * insieme 2 *',
        '* parla * cerchio due * cerchio uno *',
        '* parla * insieme due * insieme uno *',
        '* parla * cerchio 2 * cerchio 1 *',
        '* parla * insieme 2 * insieme 1 *',

        '* spiagami * dominio * codominio *',
        '* spiegami * codominio * dominio *',
        '* spiegami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* spiegami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* spiegami * insiemi *',
        '* spiegami * cerchi *',
        '* spiegami * primo cerchio * secondo cerchio *',
        '* spiegami * primo insieme * secondo insieme *',
        '* spiegami * cerchio uno * cerchio due *',
        '* spiegami * insieme uno * insieme due *',
        '* spiegami * cerchio 1 * cerchio 2 *',
        '* spiegami * insieme 1 * insieme 2 *',
        '* spiegami * cerchio due * cerchio uno *',
        '* spiegami * insieme due * insieme uno *',
        '* spiegami * cerchio 2 * cerchio 1 *',
        '* spiegami * insieme 2 * insieme 1 *',

        '* spiega * dominio * codominio *',
        '* spiega * codominio * dominio *',
        '* spiega * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* spiega * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* spiega * insiemi *',
        '* spiega * cerchi *',
        '* spiega * primo cerchio * secondo cerchio *',
        '* spiega * primo insieme * secondo insieme *',
        '* spiega * cerchio uno * cerchio due *',
        '* spiega * insieme uno * insieme due *',
        '* spiega * cerchio 1 * cerchio 2 *',
        '* spiega * insieme 1 * insieme 2 *',
        '* spiega * cerchio due * cerchio uno *',
        '* spiega * insieme due * insieme uno *',
        '* spiega * cerchio 2 * cerchio 1 *',
        '* spiega * insieme 2 * insieme 1 *',

        '* descrivimi * dominio * codominio *',
        '* descrivimi * codominio * dominio *',
        '* descrivimi * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* descrivimi * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* descrivimi * insiemi *',
        '* descrivimi * cerchi *',
        '* descrivimi * primo cerchio * secondo cerchio *',
        '* descrivimi * primo insieme * secondo insieme *',
        '* descrivimi * cerchio uno * cerchio due *',
        '* descrivimi * insieme uno * insieme due *',
        '* descrivimi * cerchio 1 * cerchio 2 *',
        '* descrivimi * insieme 1 * insieme 2 *',
        '* descrivimi * cerchio due * cerchio uno *',
        '* descrivimi * insieme due * insieme uno *',
        '* descrivimi * cerchio 2 * cerchio 1 *',
        '* descrivimi * insieme 2 * insieme 1 *',

        '* descrivi * dominio * codominio *',
        '* descrivi * codominio * dominio *',
        '* descrivi * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* descrivi * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* descrivi * insiemi *',
        '* descrivi * cerchi *',
        '* descrivi * primo cerchio * secondo cerchio *',
        '* descrivi * primo insieme * secondo insieme *',
        '* descrivi * cerchio uno * cerchio due *',
        '* descrivi * insieme uno * insieme due *',
        '* descrivi * cerchio 1 * cerchio 2 *',
        '* descrivi * insieme 1 * insieme 2 *',
        '* descrivi * cerchio due * cerchio uno *',
        '* descrivi * insieme due * insieme uno *',
        '* descrivi * cerchio 2 * cerchio 1 *',
        '* descrivi * insieme 2 * insieme 1 *',

        '* raccontami * dominio * codominio *',
        '* raccontami * codominio * dominio *',
        '* raccontami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* raccontami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* raccontami * insiemi *',
        '* raccontami * cerchi *',
        '* raccontami * primo cerchio * secondo cerchio *',
        '* raccontami * primo insieme * secondo insieme *',
        '* raccontami * cerchio uno * cerchio due *',
        '* raccontami * insieme uno * insieme due *',
        '* raccontami * cerchio 1 * cerchio 2 *',
        '* raccontami * insieme 1 * insieme 2 *',
        '* raccontami * cerchio due * cerchio uno *',
        '* raccontami * insieme due * insieme uno *',
        '* raccontami * cerchio 2 * cerchio 1 *',
        '* raccontami * insieme 2 * insieme 1 *',

        '* racconta * dominio * codominio *',
        '* racconta * codominio * dominio *',
        '* racconta * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
        '* racconta * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
        '* racconta * insiemi *',
        '* racconta * cerchi *',
        '* racconta * primo cerchio * secondo cerchio *',
        '* racconta * primo insieme * secondo insieme *',
        '* racconta * cerchio uno * cerchio due *',
        '* racconta * insieme uno * insieme due *',
        '* racconta * cerchio 1 * cerchio 2 *',
        '* racconta * insieme 1 * insieme 2 *',
        '* racconta * cerchio due * cerchio uno *',
        '* racconta * insieme due * insieme uno *',
        '* racconta * cerchio 2 * cerchio 1 *',
        '* racconta * insieme 2 * insieme 1 *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'request',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio + '-' + nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += 'Gli insiemi nel nostro esempio sono 2. Il primo insieme, l\'insieme ' + nome_insieme_dominio + ', detto in questo '
    text += 'caso dominio, è composto da ' + str(len(elementi_dominio)) + ' elementi: '
    for elemento in elementi_dominio:
        text += elemento + ', '
    text = text[:-2] + '.'

    text += ' Il secondo insieme, l\'insieme ' + nome_insieme_codominio + ', noto come codominio, è invece composto '
    text += 'da ' + str(len(elementi_codominio)) + ' elementi: '
    for elemento in elementi_codominio:
        text += elemento + ', '
    text = text[:-2] + '.'

    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        requestAnswer[domanda_uppercase] = text

    return requestAnswer

#### 4. Descrivi la funzione (NON ATTIVA)

In [245]:
def generateRequest4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    requestAnswer = {}

    domande = [
        '* descrivimi * funzione *',
        '* descrivimi * funzioni *',
        '* descrivimi * f *',
        '* descrivimi * relazione *',
        '* descrivimi * relazioni *',
        '* descrivimi * collegamento *',
        '* descrivimi * collegamenti *',
        '* descrivimi * associazione *',
        '* descrivimi * associazioni *',
        
        '* descrivi * funzione *',
        '* descrivi * funzioni *',
        '* descrivi * f *',
        '* descrivi * relazione *',
        '* descrivi * relazioni *',
        '* descrivi * collegamento *',
        '* descrivi * collegamenti *',
        '* descrivi * associazione *',
        '* descrivi * associazioni *',

        '* raccontami * funzione *',
        '* raccontami * funzioni *',
        '* raccontami * f *',
        '* raccontami * relazione *',
        '* raccontami * relazioni *',
        '* raccontami * collegamento *',
        '* raccontami * collegamenti *',
        '* raccontami * associazione *',
        '* raccontami * associazioni *',

        '* racconta * funzione *',
        '* racconta * funzioni *',
        '* racconta * f *',
        '* racconta * relazione *',
        '* racconta * relazioni *',
        '* racconta * collegamento *',
        '* racconta * collegamenti *',
        '* racconta * associazione *',
        '* racconta * associazioni *',

        '* spiegami * funzione *',
        '* spiegami * funzioni *',
        '* spiegami * f *',
        '* spiegami * relazione *',
        '* spiegami * relazioni *',
        '* spiegami * collegamento *',
        '* spiegami * collegamenti *',
        '* spiegami * associazione *',
        '* spiegami * associazioni *',

        '* spiega * funzione *',
        '* spiega * funzioni *',
        '* spiega * f *',
        '* spiega * relazione *',
        '* spiega * relazioni *',
        '* spiega * collegamento *',
        '* spiega * collegamenti *',
        '* spiega * associazione *',
        '* spiega * associazioni *',

        '* parlami * funzione *',
        '* parlami * funzioni *',
        '* parlami * f *',
        '* parlami * relazione *',
        '* parlami * relazioni *',
        '* parlami * collegamento *',
        '* parlami * collegamenti *',
        '* parlami * associazione *',
        '* parlami * associazioni *',

        '* parla * funzione *',
        '* parla * funzioni *',
        '* parla * f *',
        '* parla * relazione *',
        '* parla * relazioni *',
        '* parla * collegamento *',
        '* parla * collegamenti *',
        '* parla * associazione *',
        '* parla * associazioni *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'request',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': 'none',
        'topic': 'funzione',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += 'La funzione associa gli elementi dell\'insieme ' + nome_insieme_dominio + ' con gli elementi dell\'insieme '
    text += nome_insieme_codominio + '. In particolare, '

    for componente in link:
        text += 'l\'elemento ' + componente + ' dell\'insieme ' + nome_insieme_dominio
        text += ' è associato all\'elemento ' + link[componente] + ' dell\'insieme ' + nome_insieme_codominio + ', '
    text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    for componente in link:
        text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + componente + '-' + link[componente] + '</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        requestAnswer[domanda_uppercase] = text

    return requestAnswer

#### 5. come sono collegati gli elementi tra loro?

In [246]:
def generateRequest5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    requestAnswer = {}

    for elemento in link:
        domande = [
            '* tra di loro *',
            '* collegati * gli elementi',
            '* associati * gli elementi',
            '* elementi * collegati * frecce',
            '* ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
            '* ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
            '* relazione *',
            '* relazioni *',
            '* collegamento *',
            '* collegamenti *',
            '* associazione *',
            '* associazioni *',
            '* collegati *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'request',
            'elemento1': 'none',
            'elemento2': 'none',
            'insieme': 'none',
            'topic': 'funzione',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        for elemento in link:
            # contenuto testuale
            text += ' l\'elemento ' + elemento + ' è collegato all\'elemento ' + link[elemento] + ','
            
        # remove last comma
        text = text[:-1]
        # inserisci lettera maiscuola iniziale
        text = text[0].upper() + text[1:]

        for elemento in link:
            text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + link[elemento] + '</svgElement>'

        # immagine
        text += '<image>insiemi/' + filename + '</image>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            requestAnswer[domanda_uppercase] = text

    return requestAnswer

#### Creazione delle request

In [247]:
def generateRequest(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    setRequestAnswer = {}

    setRequestAnswer.update(generateRequest1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setRequestAnswer.update(generateRequest2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setRequestAnswer.update(generateRequest3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    #setRequestAnswer.update(generateRequest4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    setRequestAnswer.update(generateRequest5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))

    return setRequestAnswer

### Ta:propositionalQuestion

Esempio:
- L'elemento a è collegato all'elemento b?
- Esiste c nel dominio?
- ...

#### 1. Elemento a è associato all'elemento b?

In [248]:
def generatePropositionalQuestion1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    for elemento in elementi_dominio:
        for elemento2 in elementi_codominio:
            domande = [
                '* ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * associato * ' + elemento2 + ' *',
                '* ' + elemento + ' * associati * ' + elemento2 + ' *',
                '* associato * ' + elemento + ' * ' + elemento2 + ' *',
                '* associati * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * associato *',
                '* ' + elemento + ' * ' + elemento2 + ' * associati *',

                '* ' + elemento + ' * collegato * ' + elemento2 + ' *',
                '* ' + elemento + ' * collegati * ' + elemento2 + ' *',
                '* collegato * ' + elemento + ' * ' + elemento2 + ' *',
                '* collegati * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * collegato *',
                '* ' + elemento + ' * ' + elemento2 + ' * collegati *',

                '* ' + elemento + ' * relazionato * ' + elemento2 + ' *',
                '* ' + elemento + ' * relazionati * ' + elemento2 + ' *',
                '* relazionato * ' + elemento + ' * ' + elemento2 + ' *',
                '* relazionati * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * relazionato *',
                '* ' + elemento + ' * ' + elemento2 + ' * relazionati *',

                '* ' + elemento + ' * relazione * ' + elemento2 + ' *',
                '* ' + elemento + ' * relazioni * ' + elemento2 + ' *',
                '* relazione * ' + elemento + ' * ' + elemento2 + ' *',
                '* relazioni * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * relazione *',
                '* ' + elemento + ' * ' + elemento2 + ' * relazioni *',

                '* ' + elemento + ' * collegamento * ' + elemento2 + ' *',
                '* ' + elemento + ' * collegamenti * ' + elemento2 + ' *',
                '* collegamento * ' + elemento + ' * ' + elemento2 + ' *',
                '* collegamenti * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * collegamento *',
                '* ' + elemento + ' * ' + elemento2 + ' * collegamenti *',

                '* ' + elemento + ' * funzione * ' + elemento2 + ' *',
                '* ' + elemento + ' * funzioni * ' + elemento2 + ' *',
                '* funzione * ' + elemento + ' * ' + elemento2 + ' *',
                '* funzioni * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * funzione *',
                '* ' + elemento + ' * ' + elemento2 + ' * funzioni *',

                '* ' + elemento + ' * f * ' + elemento2 + ' *',
                '* ' + elemento + ' * f * ' + elemento2 + ' *',
                '* f * ' + elemento + ' * ' + elemento2 + ' *',
                '* f * ' + elemento + ' * ' + elemento2 + ' *',
                '* ' + elemento + ' * ' + elemento2 + ' * f *',
                '* ' + elemento + ' * ' + elemento2 + ' * f *',



                '* ' + elemento2 + ' * associata * ' + elemento + ' *',
                '* ' + elemento2 + ' * associato * ' + elemento + ' *',
                '* ' + elemento2 + ' * associati * ' + elemento + ' *',
                '* associata * ' + elemento2 + ' * ' + elemento + ' *',
                '* associato * ' + elemento2 + ' * ' + elemento + ' *',
                '* associati * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * associata *',
                '* ' + elemento2 + ' * ' + elemento + ' * associato *',
                '* ' + elemento2 + ' * ' + elemento + ' * associati *',

                '* ' + elemento2 + ' * collegata * ' + elemento + ' *',
                '* ' + elemento2 + ' * collegato * ' + elemento + ' *',
                '* ' + elemento2 + ' * collegati * ' + elemento + ' *',
                '* collegata * ' + elemento2 + ' * ' + elemento + ' *',
                '* collegato * ' + elemento2 + ' * ' + elemento + ' *',
                '* collegati * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * collegata *',
                '* ' + elemento2 + ' * ' + elemento + ' * collegato *',
                '* ' + elemento2 + ' * ' + elemento + ' * collegati *',

                '* ' + elemento2 + ' * relazionato * ' + elemento + ' *',
                '* ' + elemento2 + ' * relazionati * ' + elemento + ' *',
                '* relazionato * ' + elemento2 + ' * ' + elemento + ' *',
                '* relazionati * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * relazionato *',
                '* ' + elemento2 + ' * ' + elemento + ' * relazionati *',

                '* ' + elemento2 + ' * relazione * ' + elemento + ' *',
                '* ' + elemento2 + ' * relazioni * ' + elemento + ' *',
                '* relazione * ' + elemento2 + ' * ' + elemento + ' *',
                '* relazioni * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * relazione *',
                '* ' + elemento2 + ' * ' + elemento + ' * relazioni *',

                '* ' + elemento2 + ' * collegamento * ' + elemento + ' *',
                '* ' + elemento2 + ' * collegamenti * ' + elemento + ' *',
                '* collegamento * ' + elemento2 + ' * ' + elemento + ' *',
                '* collegamenti * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * collegamento *',
                '* ' + elemento2 + ' * ' + elemento + ' * collegamenti *',

                '* ' + elemento2 + ' * funzione * ' + elemento + ' *',
                '* ' + elemento2 + ' * funzioni * ' + elemento + ' *',
                '* funzione * ' + elemento2 + ' * ' + elemento + ' *',
                '* funzioni * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * funzione *',
                '* ' + elemento2 + ' * ' + elemento + ' * funzioni *',

                '* ' + elemento2 + ' * f * ' + elemento + ' *',
                '* ' + elemento2 + ' * f * ' + elemento + ' *',
                '* f * ' + elemento2 + ' * ' + elemento + ' *',
                '* f * ' + elemento2 + ' * ' + elemento + ' *',
                '* ' + elemento2 + ' * ' + elemento + ' * f *',
                '* ' + elemento2 + ' * ' + elemento + ' * f *',

            ]

            # variabili
            vars = {
                'dialogue_act': 'propositionalQuestion',
                'elemento1': elemento,
                'elemento2': elemento2,
                'insieme': 'none',
                'topic': 'funzione',
            }

            text = ''
            for var in vars:
                text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

            # contenuto testuale
            if elemento in link and elemento2 == link[elemento]:
                text += elemento + ' è collegato a ' + elemento2 + '.'
            else:
                text += elemento + ' non è collegato a ' + elemento2 + '.'

            # immagine
            if elemento2 in link and elemento2 == link[elemento]:
                text += '<image>insiemi/' + filename + '</image>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + elemento2 + '</svgElement>'
            else:
                text += '<image>insiemi/' + filename + '</image>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

            for domanda in domande:
                domanda_uppercase = domanda.upper()
                propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 2. Elemento a è nel dominio? SI

In [249]:
def generatePropositionalQuestion2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    for elemento in elementi_dominio:
        domande = [
            '* ' + elemento + ' * ' + nome_insieme_dominio + ' *',
            '* ' + elemento + ' * dominio *',
            '* ' + elemento + ' * cerchio uno *',
            '* ' + elemento + ' * insieme uno *',
            '* ' + elemento + ' * cerchio 1 *',
            '* ' + elemento + ' * insieme 1 *',
            '* ' + elemento + ' * primo cerchio *',
            '* ' + elemento + ' * primo insieme *',

            '* insieme ' + nome_insieme_dominio + ' * ' + elemento + ' *',
            '* cerchio ' + nome_insieme_dominio + ' * ' + elemento + ' *',
            '* dominio * ' + elemento + ' *',
            '* cerchio uno * ' + elemento + ' *',
            '* insieme uno * ' + elemento + ' *',
            '* cerchio 1 * ' + elemento + ' *',
            '* insieme 1 * ' + elemento + ' *',
            '* primo cerchio * ' + elemento + ' *',
            '* primo insieme * ' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'propositionalQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_dominio,
            'topic': 'insieme',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        text += '' + elemento + ' è nell\'insieme ' + nome_insieme_dominio + '.'

        # immagine
        text += '<image>insiemi/' + filename + '</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 3. Elemento a nel dominio? NO

In [250]:
def generatePropositionalQuestion3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    for elemento in elementi_codominio:
        domande = [
            '* ' + elemento + ' * insieme ' + nome_insieme_dominio + ' *',
            '* ' + elemento + ' * cerchio ' + nome_insieme_dominio + ' *',
            '* ' + elemento + ' * dominio *',
            '* ' + elemento + ' * cerchio uno *',
            '* ' + elemento + ' * insieme uno *',
            '* ' + elemento + ' * cerchio 1 *',
            '* ' + elemento + ' * insieme 1 *',
            '* ' + elemento + ' * primo cerchio *',
            '* ' + elemento + ' * primo insieme *',

            '* insieme ' + nome_insieme_dominio + ' * ' + elemento + ' *',
            '* cerchio ' + nome_insieme_dominio + ' * ' + elemento + ' *',
            '* dominio * ' + elemento + ' *',
            '* cerchio uno * ' + elemento + ' *',
            '* insieme uno * ' + elemento + ' *',
            '* cerchio 1 * ' + elemento + ' *',
            '* insieme 1 * ' + elemento + ' *',
            '* primo cerchio * ' + elemento + ' *',
            '* primo insieme * ' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'propositionalQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_codominio,
            'topic': 'insieme',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        text += elemento + ' non è nell\'insieme ' + nome_insieme_dominio + '.'
        
        # immagine
        text += '<image>insiemi/' + filename + '</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 4. Elemento a nel codominio? SI

In [251]:
def generatePropositionalQuestion4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    for elemento in elementi_codominio:
        domande = [
            '* ' + elemento + ' * ' + nome_insieme_codominio + ' *',
            '* ' + elemento + ' * codominio *',
            '* ' + elemento + ' * cerchio due *',
            '* ' + elemento + ' * insieme due *',
            '* ' + elemento + ' * cerchio 2 *',
            '* ' + elemento + ' * insieme 2 *',
            '* ' + elemento + ' * secondo cerchio *',
            '* ' + elemento + ' * secondo insieme *',

            '* insieme ' + nome_insieme_codominio + ' * ' + elemento + ' *',
            '* cerchio ' + nome_insieme_codominio + ' * ' + elemento + ' *',
            '* codominio * ' + elemento + ' *',
            '* cerchio due * ' + elemento + ' *',
            '* insieme due * ' + elemento + ' *',
            '* cerchio 2 * ' + elemento + ' *',
            '* insieme 2 * ' + elemento + ' *',
            '* secondo cerchio * ' + elemento + ' *',
            '* secondo insieme * ' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'propositionalQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_codominio,
            'topic': 'insieme',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        text += elemento + ' è nell\'insieme ' + nome_insieme_codominio + '.'

        # immagine
        text += '<image>insiemi/' + filename + '</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
        
        for domanda in domande:
            domanda_uppercase = domanda.upper()
            propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 5. Elemento a nel codominio? NO

In [252]:
def generatePropositionalQuestion5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    for elemento in elementi_dominio:
        domande = [
            '* ' + elemento + ' * insieme ' + nome_insieme_codominio + ' *',
            '* ' + elemento + ' * cerchio ' + nome_insieme_codominio + ' *',
            '* ' + elemento + ' * codominio *',
            '* ' + elemento + ' * cerchio due *',
            '* ' + elemento + ' * insieme due *',
            '* ' + elemento + ' * cerchio 2 *',
            '* ' + elemento + ' * insieme 2 *',
            '* ' + elemento + ' * secondo cerchio *',
            '* ' + elemento + ' * secondo insieme *',

            '* insieme ' + nome_insieme_codominio + ' * ' + elemento + ' *',
            '* cerchio ' + nome_insieme_codominio + ' * ' + elemento + ' *',
            '* codominio * ' + elemento + ' *',
            '* cerchio due * ' + elemento + ' *',
            '* insieme due * ' + elemento + ' *',
            '* cerchio 2 * ' + elemento + ' *',
            '* insieme 2 * ' + elemento + ' *',
            '* secondo cerchio * ' + elemento + ' *',
            '* secondo insieme * ' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'propositionalQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_codominio,
            'topic': 'insieme',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        text += elemento + ' non è nell\'insieme ' + nome_insieme_codominio + '.'

        # immagine
        text += '<image>insiemi/' + filename + '</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 6. dal primo cerchio partono freccie da tutti gli elementi?

In [253]:
def generatePropositionalQuestion6(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* primo cerchio * tutti * elementi *',
        '* primo insieme * tutti * elementi *',
        '* dominio * tutti * elementi *',
        '* ' + nome_insieme_dominio + ' * tutti * elementi *',
        '* ogni elemento * una freccia *',
        '* ogni elemento * un collegamento *',
        '* ogni elemento * un elemento *',
        '* ogni elemento * frecce *',
        '* ogni elemento * collegamenti *',
        '* ogni elemento * elementi *',

        '* tutti * elementi * primo cerchio *',
        '* tutti * elementi * primo insieme *',
        '* tutti * elementi * dominio *',
        '* tutti * elementi * insieme ' + nome_insieme_dominio + ' *',
        '* tutti * elementi * cerchio ' + nome_insieme_dominio + ' *',
        '* una freccia * ogni elemento *',
        '* un collegamento * ogni elemento *',
        '* un elemento * ogni elemento *',
        '* frecce * ogni elemento *',
        '* collegamenti * ogni elemento *',
        '* elementi * ogni elemento *',
]

    flag = True
    el_focus = []
    for elemento in elementi_dominio:
        if elemento not in link:
            flag = False
            el_focus.append(elemento)

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Da tutti gli elementi del primo cerchio (nel dominio) parte almeno una freccia.'
    else:
        text += 'No, da alcuni elementi non parte alcuna freccia, ovvero: '
        for elemento in el_focus:
            text += elemento + ', '
        text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 7. a tutti gli elementi del secondo cerchio arriva almeno una freccia?

In [254]:
def generatePropositionalQuestion7(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * secondo cerchio * arriva * freccia *',
        '* elementi * secondo insieme * arriva * freccia *',

        '* elementi * secondo cerchio * arrivano * frecce *',
        '* elementi * secondo insieme * arrivano * frecce *',

        '* elementi * secondo cerchio * collegato * freccia *',
        '* elementi * secondo insieme * collegato * freccia *',

        '* elementi * secondo cerchio * collegati * frecce *',
        '* elementi * secondo insieme * collegati * frecce *',

        '* elementi * secondo cerchio * associato * freccia *',
        '* elementi * secondo insieme * associato * freccia *',

        '* elementi * secondo cerchio * associati * frecce *',
        '* elementi * secondo insieme * associati * frecce *',

        
        '* elementi * insieme ' + nome_insieme_codominio + ' * arriva * freccia *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * arriva * freccia *',

        '* elementi * insieme ' + nome_insieme_codominio + ' * arrivano * frecce *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * arrivano * frecce *',

        '* elementi * insieme ' + nome_insieme_codominio + ' * collegato * freccia *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * collegato * freccia *',

        '* elementi * insieme ' + nome_insieme_codominio + ' * collegati * frecce *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * collegati * frecce *',

        '* elementi * insieme ' + nome_insieme_codominio + ' * associato * freccia *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * associato * freccia *',

        '* elementi * insieme ' + nome_insieme_codominio + ' * associati * frecce *',
        '* elementi * cerchio ' + nome_insieme_codominio + ' * associati * frecce *',


        '* elementi * codominio * arriva * freccia *',
        '* elementi * codominio * arrivano * frecce *',
        '* elementi * codominio * collegato * freccia *',
        '* elementi * codominio * collegati * frecce *',
        '* elementi * codominio * associato * freccia *',
        '* elementi * codominio * associati * frecce *',


        '* arriva * freccia * elemento * secondo cerchio *',
        '* arriva * freccia * elemento * secondo insieme *',

        '* arrivano * frecce * elemento * secondo cerchio *',
        '* arrivano * frecce * elemento * secondo insieme *',

        '* collegato * elemento * secondo cerchio * freccia *',
        '* collegato * elemento * secondo insieme * freccia *',

        '* collegati * frecce * elemento * secondo cerchio *',
        '* collegati * frecce * elemento * secondo insieme *',

        '* associato * freccia * elemento * secondo cerchio *',
        '* associato * freccia * elemento * secondo insieme *',

        '* associati * frecce * elemento * secondo cerchio *',
        '* associati * frecce * elemento * secondo insieme *',


        '* arriva * freccia * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* arriva * freccia * elemento * cerchio ' + nome_insieme_codominio + ' *',

        '* arrivano * frecce * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* arrivano * frecce * elemento * cerchio ' + nome_insieme_codominio + ' *',

        '* collegato * freccia * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* collegato * freccia * elemento * cerchio ' + nome_insieme_codominio + ' *',

        '* collegati * frecce * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* collegati * frecce * elemento * cerchio ' + nome_insieme_codominio + ' *',

        '* associato * freccia * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* associato * freccia * elemento * cerchio ' + nome_insieme_codominio + ' *',

        '* associati * frecce * elemento * insieme ' + nome_insieme_codominio + ' *',
        '* associati * frecce * elemento * cerchio ' + nome_insieme_codominio + ' *',


        '* arriva * freccia * elemento * codominio *',
        '* arrivano * frecce * elemento * codominio *',

        '* collegato * freccia * elemento * codominio *',
        '* collegati * frecce * elemento * codominio *',

        '* associato * freccia * elemento * codominio *',
        '* associati * frecce * elemento * codominio *',
    ]

    link_values = link.values()
    unique_values = set()  # Nuovo set per tenere traccia dei valori univoci

    for link_value in link_values:
        values = link_value.split(' ')
        for val in values:
            if val not in unique_values:
                unique_values.add(val)

    flag = True
    el_cod_focus = []
    for elemento in elementi_codominio:
        if elemento not in unique_values:
            flag = False
            el_cod_focus.append(elemento)

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_codominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Ad ogni elemento del secondo cerchio (nel codominio) arriva almeno una freccia.'
    else:
        text += 'No, ad alcuni elementi del secondo cerchio (nel codominio) non arriva nessuna una freccia, tra cui: '
        for elemento in el_cod_focus:
            text += elemento + ', '
        text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 8. ci sono elementi del primo cerchio da cui parte più di una freccia?

In [255]:
def generatePropositionalQuestion8(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * primo cerchio * piu * freccia *',
        '* elementi * primo insieme * piu * freccia *',
        '* elementi * dominio * piu * freccia *',
        '* elementi * insieme ' + nome_insieme_dominio + ' * piu * freccia *',
        '* da ogni elemento * primo cerchio * piu * freccia *',
        '* da ogni elemento * primo insieme * piu * freccia *',
        '* da ogni elemento * dominio * piu * freccia *',
        '* da ogni elemento * insieme ' + nome_insieme_dominio + ' * piu * freccia *',
        '* elementi * primo cerchio * piu * frecce *',
        '* elementi * primo insieme * piu * frecce *',
        '* elementi * dominio * piu * frecce *',
        '* elementi * insieme ' + nome_insieme_dominio + ' * piu * frecce *',
        '* da ogni elemento * primo cerchio * piu * frecce *',
        '* da ogni elemento * primo insieme * piu * frecce *',
        '* da ogni elemento * dominio * piu * frecce *',
        '* da ogni elemento * insieme ' + nome_insieme_dominio + ' * piu * frecce *',
        '* piu * freccia *',
        '* piu * frecce *',
    ]

    #controlla che in link ci sia un'unica occorrenza per ogni elemento del dominio
    elementi = {}
    for elemento in link:
        el = elemento.split('_')[0]
        if el in elementi:
            elementi[el] += 1
        else:
            elementi[el] = 1

    #controlla se c'è una chiave con valore > 1
    flag = False
    elementi_piu_di_una_freccia = []
    for el in elementi:
        if elementi[el] > 1:
            flag = True
            elementi_piu_di_una_freccia.append(el)

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Ad ogni elemento del primo cerchio (nel dominio) parte una freccia, ovvero: '
        for elemento in elementi_piu_di_una_freccia:
            text += elemento + ', '
        text = text[:-2] + '.'
    else:
        text += 'No, da ogni elmento del primo cerchio parte al massimo una freccia.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 9. ci sono elementi del secondo cerchio a cui arriva più di una freccia?

In [256]:
def generatePropositionalQuestion9(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * secondo cerchio * piu * freccia *',
        '* elementi * secondo insieme * piu * freccia *',
        '* elementi * codominio * piu * freccia *',
        '* elementi * ' + nome_insieme_codominio + ' * piu * freccia *',
        '* da ogni elemento * secondo cerchio * piu * freccia *',
        '* da ogni elemento * secondo insieme * piu * freccia *',
        '* da ogni elemento * codominio * piu * freccia *',
        '* da ogni elemento * ' + nome_insieme_codominio + ' * piu * freccia *',
        '* elementi * secondo cerchio * piu * frecce *',
        '* elementi * secondo insieme * piu * frecce *',
        '* elementi * codominio * piu * frecce *',
        '* elementi * ' + nome_insieme_codominio + ' * piu * frecce *',
        '* da ogni elemento * secondo cerchio * piu * frecce *',
        '* da ogni elemento * secondo insieme * piu * frecce *',
        '* da ogni elemento * codominio * piu * frecce *',
        '* da ogni elemento * ' + nome_insieme_codominio + ' * piu * frecce *',
    ]

    # conta frecce che arrivano agli elementi del codominio
    el_codomonio = {}
    for key in link:
        if link[key] in el_codomonio:
            el_codomonio[link[key]] += 1
        else:
            el_codomonio[link[key]] = 1

    #controlla se c'è una chiave con valore > 1
    flag = False
    for el in el_codomonio:
        if el_codomonio[el] > 1:
            flag = True


    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Si, '
        for elemento in el_codomonio:
            if el_codomonio[elemento] > 1:
                text += 'a ' + elemento + ' arrivano '
                text += str(el_codomonio[elemento]) + ' frecce, '

        text = text[:-2] + '.'
    else:
        text += 'Ad ogni elemento del secondo cerchio (nel codominio) arriva una freccia.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### 10. quali elementi di B sono collegati a tutti gli elementi di A?

In [257]:
def generatePropositionalQuestion10(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * secondo cerchio * tutti * elementi *',
        '* elementi * secondo insieme * tutti * elementi *',
        '* elementi * codominio * tutti * elementi *',
        '* elementi * ' + nome_insieme_codominio + ' * tutti * elementi *',

        '* elementi * secondo cerchio * ogni * elemento *',
        '* elementi * secondo insieme * ogni * elemento *',
        '* elementi * codominio * ogni * elemento *',
        '* elementi * ' + nome_insieme_codominio + ' * ogni * elemento *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    text = ''
    for elemento in elementi_codominio:
        flag = True
        for link_value in link.values():
            values = link_value.split(' ')
            if elemento not in values:
                flag = False
                break

        if flag:
            if text == '':
                text += 'Gli elementi del secondo cerchio collegati a tutti gli elementi del primo cerchio sono: '
            text += elemento + ', '

    if text == '':
        text = 'Nessun elemento del secondo cerchio è collegato a tutti gli elementi del primo cerchio.'
    else:
        text = text[:-2] + '.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

### Elementi di B senza freccia

In [258]:
def generatePropositionalQuestion11(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * secondo * senza * freccia *',
        '* elementi * codominio * senza * freccia *',
        '* elementi * ' + nome_insieme_codominio + ' * senza * freccia *',
        '* elementi * secondo * senza * frecce *',
        '* elementi * codominio * senza * frecce *',
        '* elementi * ' + nome_insieme_codominio + ' * senza * frecce *',
        '* elementi * secondo * nessuna * freccia *',
        '* elementi * codominio * nessuna * freccia *',
        '* elementi * ' + nome_insieme_codominio + ' * nessuna * freccia *',
        '* elementi * secondo * nessuna * frecce *',
        '* elementi * codominio * nessuna * frecce *',

        '* elemento * secondo * senza * freccia *',
        '* elemento * codominio * senza * freccia *',
        '* elemento * ' + nome_insieme_codominio + ' * senza * freccia *',
        '* elemento * secondo * senza * frecce *',
        '* elemento * codominio * senza * frecce *',
        '* elemento * ' + nome_insieme_codominio + ' * senza * frecce *',
        '* elemento * secondo * nessuna * freccia *',
        '* elemento * codominio * nessuna * freccia *',
        '* elemento * ' + nome_insieme_codominio + ' * nessuna * freccia *',
        '* elemento * secondo * nessuna * frecce *',
        '* elemento * codominio * nessuna * frecce *',
    ]

    # conta frecce che arrivano agli elementi del codominio
    el_codomonio = {}
    for key in elementi_codominio:
        el_codomonio[key] = 0

    for key in link:
        if link[key] in el_codomonio:
            el_codomonio[link[key]] += 1
        else:
            el_codomonio[link[key]] = 1

    #controlla se c'è una chiave con valore = 0
    flag = False
    for el in el_codomonio:
        if el_codomonio[el] == 0:
            flag = True


    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Si, i seguenti elementi del secondo cerchio (nel codominio) non sono collegati a nessun elemento del primo cerchio: '
        for elemento in elementi_codominio:
            if el_codomonio[elemento] == 0:
                text += elemento + ', '

        text = text[:-2] + '.'
    else:
        text += 'No, tutti gli elementi del secondo cerchio (nel codominio) sono collegati ad almeno un elemento del primo cerchio.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    
    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

### Elementi di A senza freccia

In [259]:
def generatePropositionalQuestion12(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    domande = [
        '* elementi * primo * senza * freccia *',
        '* elementi * dominio * senza * freccia *',
        '* elementi * ' + nome_insieme_dominio + ' * senza * freccia *',
        '* elementi * primo * non * freccia *',
        '* elementi * dominio * non * freccia *',
        '* elementi * ' + nome_insieme_dominio + ' * non * freccia *',
        '* elementi * primo * senza * frecce *',
        '* elementi * dominio * senza * frecce *',
        '* elementi * ' + nome_insieme_dominio + ' * senza * frecce *',
        '* elementi * primo * non * frecce *',
        '* elementi * dominio * non * frecce *',
        '* elementi * ' + nome_insieme_dominio + ' * non * frecce *',
        '* elementi * primo * nessuna * freccia *',
        '* elementi * dominio * nessuna * freccia *',
        '* elementi * ' + nome_insieme_dominio + ' * nessuna * freccia *',
        '* elementi * primo * nessuna * frecce *',
        '* elementi * dominio * nessuna * frecce *',

        '* elemento * primo * senza * freccia *',
        '* elemento * dominio * senza * freccia *',
        '* elemento * ' + nome_insieme_dominio + ' * senza * freccia *',
        '* elemento * primo * senza * frecce *',
        '* elemento * dominio * senza * frecce *',
        '* elemento * ' + nome_insieme_dominio + ' * senza * frecce *',
        '* elemento * primo * nessuna * freccia *',
        '* elemento * dominio * nessuna * freccia *',
        '* elemento * ' + nome_insieme_dominio + ' * nessuna * freccia *',
        '* elemento * primo * nessuna * frecce *',
        '* elemento * dominio * nessuna * frecce *',
    ]

    # conta frecce che arrivano agli elementi del codominio
    el_dominio = {}
    for key in elementi_dominio:
        el_dominio[key] = 0

    for key in link:
        if key in el_dominio:
            el_dominio[key] += 1
        else:
            el_dominio[key] = 1

    #controlla se c'è una chiave con valore = 0
    flag = False
    for el in el_dominio:
        if el_dominio[el] == 0:
            flag = True


    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': 'none',
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

    # contenuto testuale
    if flag:
        text += 'Si, i seguenti elementi del primo cerchio (nel dominio) non sono collegati a nessun elemento del secondo cerchio: '
        for elemento in elementi_dominio:
            if el_dominio[elemento] == 0:
                text += elemento + ', '

        text = text[:-2] + '.'
    else:
        text += 'No, tutti gli elementi del primo cerchio (nel dominio) sono collegati ad almeno un elemento del primo secondo.'

    # immagine
    text += '<image>insiemi/' + filename + '</image>'
    
    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text

    return propositionalQuestionAnswer

#### Creazione propositionalQuestion

In [260]:
def generatePropositionalQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    propositionalQuestionAnswer = {}

    propositionalQuestionAnswer.update(generatePropositionalQuestion1(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion2(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion3(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion4(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion5(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion6(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion7(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion8(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion9(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion10(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion11(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    propositionalQuestionAnswer.update(generatePropositionalQuestion12(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))

    return propositionalQuestionAnswer

### Domanda non compresa

In [261]:
def generateNotUnderstoodQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link):
    notUnderstoodAnser = {
        '*': 'Perdonami non ho capito. Potresti provare a riformulare la domanda utilizzando parole chiavi come insieme, collegato, associato o nominando qualche elemento degli insiemi?'
    }

    return notUnderstoodAnser

## Costruzione dei file aiml

In [262]:
def build_aiml(directory_name, dialogue_acts, setQuestionAnswer, requestAnswer, propositionalQuestionAnswer):
    #for dialogue_act in dialogue_acts:
    file_path = 'aiml/' + directory_name + '/' + directory_name + '.aiml'
    root = ET.Element('aiml')

    #if dialogue_act == 'setQuestionAnswer':
    #    risposte = setQuestionAnswer
    #elif dialogue_act == 'requestAnswer':
    #    risposte = requestAnswer
    #elif dialogue_act == 'propositionalQuestionAnswer':
    #    risposte = propositionalQuestionAnswer
    #else:
    #    break

    #merge setQuestionAnswer, requestAnswer, propositionalQuestionAnswer
    risposte = {}
    risposte.update(propositionalQuestionAnswer)
    risposte.update(requestAnswer)
    risposte.update(setQuestionAnswer)

    # ordina le risposte in ordine di lunghezza della key
    risposte = dict(sorted(risposte.items(), key=lambda item: len(item[0]), reverse=True))

    for domanda in risposte:
        #crea un tag xml chiamato category
        category = ET.Element('category')
        #inserisci all'interno un altro tag chiaamto pattern contentene un * e crea un tag chiamato template con il valore di text[0]
        pattern = ET.SubElement(category, 'pattern')
        pattern.text = domanda
        template = ET.SubElement(category, 'template')
        template.text = risposte[domanda]
        

        #aggiungi il tag category al tag root
        root.append(category)

    #salva il file xml
    tree = ET.ElementTree(root)
    tree.write(file_path)

    # Ottieni la rappresentazione del testo non escapato
    xml_str = xml.dom.minidom.parseString(ET.tostring(root)).toprettyxml(indent="    ")

    # Sovrascrivi il file AIML con le modifiche
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(xml_str)

    # Apri il file appena creato
    webbrowser.open(file_path)

    # Leggi il contenuto del file
    with open(file_path, 'r', encoding='utf-8') as file:
        file_content = file.read()

    # Sostituisci "&lt;" con "<" e "&gt;" con ">"
    file_content = file_content.replace("&lt;", "<").replace("&gt;", ">").replace("&quot;", "\"")

    # Sovrascrivi il file con le modifiche
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(file_content)

    return directory_name + ' AIML files have been created successfully!'



## MAIN

In [263]:
filenames = os.listdir('svg')

for filename in filenames:
    svg = parsing_svg('svg/' + filename)
    nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link = information_retrival_by_svg_tree(svg)

    #print('DOMINIO:', nome_insieme_dominio, elementi_dominio)
    #print('CODOMINIO:', nome_insieme_codominio, elementi_codominio)
    #print('LINK:', link)

    requestAnswer = generateRequest(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link)
    propositionalQuestionAnswer = generatePropositionalQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link)
    setQuestionAnswer = generateSetQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link)
    setQuestionAnswer.update(generateNotUnderstoodQuestion(filename, nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link))
    
    directory_name = filename.split('.')[0]

    ret = build_aiml(directory_name, ['setQuestionAnswer', 'requestAnswer', 'propositionalQuestionAnswer'], setQuestionAnswer, requestAnswer, propositionalQuestionAnswer)

### descrizione generica insiemi

In [264]:
filenames = os.listdir('svg')

for filename in filenames:
    svg = parsing_svg('svg/' + filename)
    nome_insieme_dominio, elementi_dominio, nome_insieme_codominio, elementi_codominio, link = information_retrival_by_svg_tree(svg)

    text = 'L\'esempio è composto da due insiemi: il primo insieme, l\'insieme ' + nome_insieme_dominio + ', detto in questo caso dominio, è composto da ' + str(len(elementi_dominio)) + ' elementi: '
    for elemento in elementi_dominio:
        text += elemento + ', '
    text = text[:-2] + '. Il secondo insieme, l\'insieme ' + nome_insieme_codominio + ', noto come codominio, è invece composto da ' + str(len(elementi_codominio)) + ' elementi: '
    for elemento in elementi_codominio:
        text += elemento + ', '
    text = text[:-2] + '. Gli elementi del dominio sono collegati agli elementi del codominio. In particolare, '
    for componente in link:
        el_cod = link[componente].split(' ')
        if (len(el_cod) > 1):
            el_cod_length = len(el_cod)
            text += componente + ' è associato a '
            for el in el_cod:
                if (el_cod_length == 1):
                    text += el
                elif (el_cod_length == 2):
                    text += el + ' e '
                else:
                    text += el + ', '
        else:
            text += componente + ' è associato a ' + link[componente] + ', '
    text = text[:-2] + '.'

    #salva in un file
    with open('aiml/' + filename.split('.')[0] + '/descrizione.txt', 'w', encoding='utf-8') as file:
        file.write(text)